In [0]:
import dlt
from pyspark.sql import functions as F

CATALOG = "capstone_five"

# ---------- SILVER: clean, validate, enrich ----------
@dlt.table(
    name=f"{CATALOG}.silver.vitals",
    comment="Cleaned and validated vitals, joined with PII-free patient attributes")
@dlt.expect_or_drop("valid_heart_rate", "heart_rate BETWEEN 20 AND 250")
@dlt.expect_or_drop("valid_spo2", "spo2 BETWEEN 50 AND 100")
@dlt.expect("has_patient_id", "patient_id IS NOT NULL")
def silver_vitals():
    vitals = (spark.readStream.table(f"{CATALOG}.bronze.vitals")
              .withColumn("ts", F.to_timestamp("ts"))
              .withWatermark("ts", "2 minutes")
              .dropDuplicates(["patient_id", "ts"]))
    patients = spark.read.table(f"{CATALOG}.silver.dim_patients_clinical")
    return vitals.join(patients, "patient_id", "left")

# ---------- GOLD 1: per-patient vitals per minute (dashboard) ----------
@dlt.table(
    name=f"{CATALOG}.gold.vitals_1min",
    comment="Average vitals per patient per minute")
def gold_vitals_1min():
    return (spark.read.table(f"{CATALOG}.silver.vitals")
            .groupBy(F.window("ts", "1 minute").alias("minute"), "patient_id")
            .agg(F.round(F.avg("heart_rate"), 1).alias("avg_hr"),
                 F.max("heart_rate").alias("max_hr"),
                 F.round(F.avg("glucose"), 1).alias("avg_glucose"),
                 F.min("spo2").alias("min_spo2"),
                 F.count("*").alias("readings")))

# ---------- GOLD 2: anomalies (alerts) ----------
@dlt.table(
    name=f"{CATALOG}.gold.anomalies",
    comment="Abnormal readings: tachycardia, low oxygen, high glucose")
def gold_anomalies():
    return (spark.readStream.table(f"{CATALOG}.silver.vitals")
            .withColumn("alert_type",
                F.when(F.col("heart_rate") > 100, "TACHYCARDIA")
                 .when(F.col("spo2") < 90, "LOW_SPO2")
                 .when(F.col("glucose") > 180, "HIGH_GLUCOSE"))
            .filter(F.col("alert_type").isNotNull())
            .select("patient_id", "ts", "heart_rate", "spo2", "glucose",
                    "alert_type", "GENDER", "BIRTHDATE"))